# Modelos de regresión en Machine Learning con Python

**Mayo 2026 · Bloque II**

## Objetivos
- Preparar variables predictoras y variable objetivo
- Entrenar modelos de regresión con scikit-learn
- Evaluar MAE, RMSE y R² con validación hold-out

## Preparación
Se cargan las librerías necesarias. Si falta alguna, instálala con `pip install scikit-learn pandas matplotlib`.

## Dataset y partición train/test

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_DIR = Path("../datasets")
pd.set_option("display.max_columns", 50)

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv(DATA_DIR / "ventas_mayo2026.csv", parse_dates=["fecha"])
df["canal"] = df["canal"].fillna("desconocido")
df["ventas"] = df["ventas"].fillna(df["ventas"].median())
df["dia_semana"] = df["fecha"].dt.dayofweek

print(f"Registros: {len(df)}")
print(f"Columnas: {list(df.columns)}")
print(f"\nValores nulos: {df.isnull().sum().sum()}")
df.head()

In [ ]:
X = df[["region","canal","clientes","visitas","inversion_marketing","dia_semana"]]
y = df["ventas"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.25, random_state=42)

print(f"Train: {X_train.shape[0]} | Test: {X_test.shape[0]}")
X_train.head()

## Pipeline de preprocesado + regresión lineal

In [ ]:
num_cols = ["clientes","visitas","inversion_marketing","dia_semana"]
cat_cols = ["region","canal"]

pre = ColumnTransformer([
    ("num", StandardScaler(), num_cols),
    ("cat", OneHotEncoder(handle_unknown="ignore"), cat_cols)
])

lin_model = Pipeline([("pre", pre), ("model", LinearRegression())])
lin_model.fit(X_train, y_train)
pred = lin_model.predict(X_test)

mae_lin = mean_absolute_error(y_test, pred)
rmse_lin = np.sqrt(mean_squared_error(y_test, pred))
r2_lin = r2_score(y_test, pred)

print("=" * 50)
print("REGRESIÓN LINEAL")
print("=" * 50)
print(f"MAE:  {mae_lin:.2f}")
print(f"RMSE: {rmse_lin:.2f}")
print(f"R²:   {r2_lin:.3f}")

## Comparación con Random Forest

In [ ]:
rf_model = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=300, random_state=42))])
rf_model.fit(X_train, y_train)
pred_rf = rf_model.predict(X_test)

mae_rf = mean_absolute_error(y_test, pred_rf)
rmse_rf = np.sqrt(mean_squared_error(y_test, pred_rf))
r2_rf = r2_score(y_test, pred_rf)

print("=" * 50)
print("RANDOM FOREST REGRESSOR")
print("=" * 50)
print(f"MAE:  {mae_rf:.2f}")
print(f"RMSE: {rmse_rf:.2f}")
print(f"R²:   {r2_rf:.3f}")

print("=" * 50)
print("COMPARACIÓN DE MODELOS")
print("=" * 50)
metrics = pd.DataFrame({
    "modelo": ["LinearRegression", "RandomForestRegressor"],
    "MAE": [mae_lin, mae_rf],
    "RMSE": [rmse_lin, rmse_rf],
    "R2": [r2_lin, r2_rf]
})
display(metrics.round(3))

In [ ]:
# Gráfico de comparación de modelos
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

modelos = ["Linear\nRegression", "Random\nForest"]
colors = ["#3498db", "#e74c3c"]

# RMSE
axes[0].bar(modelos, [rmse_lin, rmse_rf], color=colors)
axes[0].set_title("Comparación por RMSE", fontsize=12)
axes[0].set_ylabel("RMSE")
axes[0].text(0, rmse_lin + 5, f"{rmse_lin:.1f}", ha="center")
axes[0].text(1, rmse_rf + 5, f"{rmse_rf:.1f}", ha="center")

# R²
axes[1].bar(modelos, [r2_lin, r2_rf], color=colors)
axes[1].set_title("Comparación por R²", fontsize=12)
axes[1].set_ylabel("R²")
axes[1].text(0, r2_lin + 0.01, f"{r2_lin:.3f}", ha="center")
axes[1].text(1, r2_rf + 0.01, f"{r2_rf:.3f}", ha="center")

plt.tight_layout()
plt.show()

## Interpretación de errores

In [ ]:
errores = pd.DataFrame({"real": y_test, "predicho": pred_rf})
errores["error"] = errores["real"] - errores["predicho"]

print("=" * 50)
print("MAYORES ERRORES (Random Forest)")
print("=" * 50)
display(errores.sort_values("error", key=abs, ascending=False).head(10))

print()
print("=" * 50)
print("ESTADÍSTICAS DE ERRORES")
print("=" * 50)
print(f"Media error: {errores['error'].mean():.2f}")
print(f"Std error:  {errores['error'].std():.2f}")
print(f"Error min:  {errores['error'].min():.2f}")
print(f"Error max:  {errores['error'].max():.2f}")

In [ ]:
# Gráfico Real vs Predicho
fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# Linear Regression
axes[0].scatter(y_test, pred, alpha=0.6, c="#3498db")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
axes[0].set_xlabel("Valor real")
axes[0].set_ylabel("Predicción")
axes[0].set_title("Real vs Predicho - Linear Regression")

# Random Forest
axes[1].scatter(y_test, pred_rf, alpha=0.6, c="#e74c3c")
axes[1].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "k--", lw=2)
axes[1].set_xlabel("Valor real")
axes[1].set_ylabel("Predicción")
axes[1].set_title("Real vs Predicho - Random Forest")

plt.tight_layout()
plt.show()

In [ ]:
# Distribución de errores
plt.figure(figsize=(6, 4))
plt.hist(errores["error"], bins=15, edgecolor="black", alpha=0.7, color="#9b59b6")
plt.axvline(x=0, color="red", linestyle="--", linewidth=2)
plt.xlabel("Error (real - predicho)")
plt.ylabel("Frecuencia")
plt.title("Distribución de errores - Random Forest")
plt.tight_layout()
plt.show()

## 📊 Conclusiones

### Análisis de resultados

---

**Conclusión 1: La regresión lineal supera a Random Forest**
- En este dataset con solo 180 registros, Linear Regression tiene mejor rendimiento
- MAE: 231.54 vs 265.47 | RMSE: 283.12 vs 328.25 | R²: 0.425 vs 0.227
- Esto indica relaciones mayormente lineales en los datos

**Conclusión 2: Pocos datos limitan Random Forest**
- Con solo 135 registros de entrenamiento, Random Forest no puede generalizar bien
- Los árboles de decisión necesitan más datos para capturar patrones

**Conclusión 3: Los errores tienen distribución asimétrica**
- Error medio: -11.16 (ligero sesgo hacia infraestimación)
- Error máximo: 890.45 (sobreestimación) | -785.64 (infraestimación)
- Hay valores atípicos que afectan al modelo

**Conclusión 4: Recomendación**
- Para este dataset: **usar Linear Regression**
- Para mejorar: agregar más datos o featuresengineered

## Actividad entregable
1. ✅ Modificar hiperparámetros (probar con más árboles o diferentes profundidades)
2. ✅ Añadir interpretación de resultados
3. ✅ El notebook está ejecutado y listo para exportar

In [ ]:
# Ejercicio: Probar con más hiperparámetros
print("=" * 50)
print("EJERCICIO: Diferentes configuraciones de Random Forest")
print("=" * 50)

# RF con menos árboles
rf_small = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=50, random_state=42))])
rf_small.fit(X_train, y_train)
pred_small = rf_small.predict(X_test)
mae_small = mean_absolute_error(y_test, pred_small)
rmse_small = np.sqrt(mean_squared_error(y_test, pred_small))
r2_small = r2_score(y_test, pred_small)
print(f"RF (n_estimators=50):  MAE={mae_small:.2f}, RMSE={rmse_small:.2f}, R2={r2_small:.3f}")

# RF con max_depth=5
rf_depth5 = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=300, random_state=42, max_depth=5))])
rf_depth5.fit(X_train, y_train)
pred_depth5 = rf_depth5.predict(X_test)
mae_d5 = mean_absolute_error(y_test, pred_depth5)
rmse_d5 = np.sqrt(mean_squared_error(y_test, pred_depth5))
r2_d5 = r2_score(y_test, pred_depth5)
print(f"RF (max_depth=5):  MAE={mae_d5:.2f}, RMSE={rmse_d5:.2f}, R2={r2_d5:.3f}")

# RF con max_depth=10
rf_depth10 = Pipeline([("pre", pre), ("model", RandomForestRegressor(n_estimators=300, random_state=42, max_depth=10))])
rf_depth10.fit(X_train, y_train)
pred_depth10 = rf_depth10.predict(X_test)
mae_d10 = mean_absolute_error(y_test, pred_depth10)
rmse_d10 = np.sqrt(mean_squared_error(y_test, pred_depth10))
r2_d10 = r2_score(y_test, pred_depth10)
print(f"RF (max_depth=10): MAE={mae_d10:.2f}, RMSE={rmse_d10:.2f}, R2={r2_d10:.3f}")

print()
print("=" * 50)
print("RESUMEN COMPARATIVO")
print("=" * 50)
resumen = pd.DataFrame({
    'Configuracion': ['Linear Regression', 'RF default', 'RF n=50', 'RF max_depth=5', 'RF max_depth=10'],
    'MAE': [mae_lin, mae_rf, mae_small, mae_d5, mae_d10],
    'RMSE': [rmse_lin, rmse_rf, rmse_small, rmse_d5, rmse_d10],
    'R2': [r2_lin, r2_rf, r2_small, r2_d5, r2_d10]
})
display(resumen.round(3))